In [ ]:
%pip install pandas numpy scikit-learn matplotlib joblib

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

project = Path.cwd()

if (project / "data" / "cars.csv").exists():
    base = project
elif (project.parent / "data" / "cars.csv").exists():
    base = project.parent
else:
    raise FileNotFoundError("cars.csv nije pronadjen.")

sys.path.append(str(base / "src"))

from data_cleaning import clean_data
from feature_engineering import add_features
from data_preprocessing import build_preprocessor, all_features


In [ ]:
df = pd.read_csv(base / "data" / "cars.csv")

print(df.shape)
print(df.head())
print()
print(df.dtypes)
print()
print(df.isna().sum())
print()
print(df["priceUSD"].describe())


In [ ]:
df = clean_data(df)
print(df.shape)
print(df.isna().sum())


In [ ]:
df = add_features(df)

print(df[[
    "make",
    "model",
    "priceUSD",
    "car_age",
    "mileage_per_year",
    "engine_volume_liters",
    "is_newer_car",
    "is_high_mileage",
    "brand_model"
]].head())


In [ ]:
X = df[all_features]
y = df["priceUSD"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=10,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=5,
        max_depth=10,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )
}

results = []
trained_models = {}

for name, estimator in models.items():
    model = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", estimator)
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mse = mean_squared_error(y_test, pred)

    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mse,
        "RMSE": mse ** 0.5,
        "R2": r2_score(y_test, pred)
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("MAE")

print(results_df)


In [ ]:
results_df.set_index("model")[["MAE", "RMSE"]].plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel("USD")
plt.title("Poredjenje modela")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = trained_models[best_name]

pred = best_model.predict(X_test)

examples = pd.DataFrame({
    "actual_price": y_test.values[:10],
    "predicted_price": pred[:10]
})

examples["error"] = (
    examples["actual_price"] -
    examples["predicted_price"]
).abs()

print("Najbolji model:", best_name)
print()
print(examples)


In [ ]:
final_model = trained_models[best_name]

final_model.fit(X, y)

joblib.dump(
    final_model,
    base / "models" / "car_price_model.joblib"
)

print("Model sacuvan.")


In [ ]:
test_car = pd.DataFrame({
    "make": ["volkswagen"],
    "model": ["golf"],
    "year": [2014],
    "condition": ["with mileage"],
    "mileage_km": [180000],
    "fuel_type": ["diesel"],
    "engine_volume_cm3": [1600],
    "color": ["black"],
    "transmission": ["mechanics"],
    "drive_unit": ["front-wheel drive"],
    "segment": ["c"]
})

test_car = add_features(test_car)

predicted_price = final_model.predict(
    test_car[all_features]
)[0]

print("Predvidjena cijena:", round(predicted_price, 2))
